# Chapter 7: Support Vector Machines and Kernel Methods

## Ringkasan Bab & Penjelasan Teoretis
**Support Vector Machines (SVM)** bekerja dengan mencari **hyperplane** yang memaksimalkan **margin** (jarak) terhadap titik data terdekat dari masing-masing kelas (*support vectors*).

### 1. Kernel Functions & *The Kernel Trick*
Untuk menangani data yang tidak terpisahkan secara linear, fungsi kernel memetakan data secara implisit ke ruang berdimensi lebih tinggi tanpa menghitung koordinat barunya secara langsung:
* **Linear Kernel**: Memisahkan data secara linear di ruang asli.
* **Polynomial Kernel**: Menggunakan fungsi polinomial berderajat `degree` untuk menangkap hubungan kurvilinear.
* **Radial Basis Function (RBF) Kernel**: Memetakan data menggunakan fungsi Gaussian untuk membentuk batas keputusan non-linear yang kompleks.

### 2. Penyetelan Parameter SVM & Ruang Dimensi Tinggi
Parameter regularisasi **$C$** mengatur kompromi antara lebar margin dan toleransi terhadap kesalahan klasifikasi (*soft margin*). Nilai $C$ tinggi mempersempit margin demi meminimalkan salah klasifikasi pada data latih.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, make_classification
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC, SVR
from sklearn.metrics import accuracy_score, classification_report, mean_squared_error

iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=2024)

# 1. SVC (Klasifikasi) dan SVR (Regresi)
svm_classifier = SVC(kernel='linear').fit(X_train, y_train)
print(f"Linear SVC Accuracy: {accuracy_score(y_test, svm_classifier.predict(X_test)):.2f}")

svm_regressor = SVR(kernel='rbf').fit(X_train[:, :2], y_train)
print(f"RBF SVR MSE: {mean_squared_error(y_test, svm_regressor.predict(X_test[:, :2])):.2f}")

# 2. Perbandingan Kernel (Linear, Polynomial, RBF) & Visualisasi Decision Boundaries
x_min, x_max = X_train[:, 0].min() - 1, X_train[:, 0].max() + 1
y_min, y_max = X_train[:, 1].min() - 1, X_train[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02), np.arange(y_min, y_max, 0.02))

X_train_2d = X_train[:, :2]
svm_linear_2d = SVC(kernel='linear').fit(X_train_2d, y_train)
svm_poly_2d = SVC(kernel='poly', degree=3).fit(X_train_2d, y_train)
svm_rbf_2d = SVC(kernel='rbf').fit(X_train_2d, y_train)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, mod, title in zip(axes, [svm_linear_2d, svm_poly_2d, svm_rbf_2d], ['Linear Kernel', 'Polynomial Kernel', 'RBF Kernel']):
    Z = mod.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.8)
    ax.scatter(X_train_2d[:, 0], X_train_2d[:, 1], c=y_train, edgecolors='k', marker='o')
    ax.set_title(title)
plt.tight_layout()
plt.show()

# 3. Tuning Parameter SVM dengan GridSearchCV
param_grid = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf', 'poly'],
    'degree': [2, 3, 4]
}
grid_search = GridSearchCV(SVC(), param_grid, cv=5).fit(X_train, y_train)
print("Best Parameters:", grid_search.best_params_)
print(f"Best CV Score: {grid_search.best_score_:.2f}")

# 4. SVM pada Ruang Berdimensi Tinggi (High-Dimensional Space)
X_hd, y_hd = make_classification(n_samples=500, n_features=200, n_informative=20, random_state=2024)
X_hd_tr, X_hd_te, y_hd_tr, y_hd_te = train_test_split(X_hd, y_hd, test_size=0.3, random_state=2024)
svm_hd = SVC(kernel='rbf', C=1.0, gamma='scale').fit(X_hd_tr, y_hd_tr)
print(f"High-Dimensional SVM Accuracy (200 features): {accuracy_score(y_hd_te, svm_hd.predict(X_hd_te)):.3f}")